# Instagram Modeling Pipeline
Builds three modeling tables (comments, users, posts) from already-computed LLM labels.
No Vertex / GCS calls. All data is local under `Ali/outputs/` and `Data/`.

In [1]:

import warnings
warnings.filterwarnings("ignore")

import pathlib
import pandas as pd
import numpy as np

# ── Paths ──────────────────────────────────────────────────────────────────
REPO = pathlib.Path("..").resolve()   # AFB_Lab root
OUT  = REPO / "Ali" / "outputs"
DATA = REPO / "Data"
MOD  = OUT  / "modeling"
MOD.mkdir(parents=True, exist_ok=True)

print("Output dir:", MOD)

# ── Source paths ───────────────────────────────────────────────────────────
PATH_B  = OUT  / "stage2_sentiment"        / "sentiment_instagram.parquet"
PATH_C  = OUT  / "stage2_persona_combined" / "user_personas_combined.parquet"
PATH_D  = OUT  / "ig_multimodal_final.parquet"
PATH_E  = DATA / "ig_posts_cleaned.parquet"

for p in [PATH_B, PATH_C, PATH_D, PATH_E]:
    status = "OK" if p.exists() else "MISSING"
    print(f"  {status}  {p.name}")


Output dir: D:\Polythecninco di Milano\AFB_Lab\Ali\outputs\modeling
  OK  sentiment_instagram.parquet
  OK  user_personas_combined.parquet
  OK  ig_multimodal_final.parquet
  OK  ig_posts_cleaned.parquet


## Load raw sources

In [2]:

# ── B: sentiment (IG-only by construction) ─────────────────────────────────
B = pd.read_parquet(PATH_B)
# Enforce string ids (guard against silent int cast)
for col in ["comment_id", "author_id", "media_id"]:
    if col in B.columns:
        B[col] = B[col].astype(str)

print("B — sentiment_instagram")
print("  shape :", B.shape)
print("  dtypes excerpt:")
print(B[["comment_id","author_id","media_id","sentiment","sentiment_score",
          "emotion","intent","target","intensity","sarcasm","toxicity",
          "sentiment_cat","lang"]].dtypes)
print()
print("  sentiment_score min/max:", B["sentiment_score"].min(), B["sentiment_score"].max())
# clip any stray values
B["sentiment_score"] = B["sentiment_score"].clip(-1, 1)

# ── C: user personas ───────────────────────────────────────────────────────
C = pd.read_parquet(PATH_C)
C["author_id"] = C["author_id"].astype(str)
print("\nC — user_personas_combined")
print("  shape :", C.shape)
print("  dtypes:", C[["author_id","persona_codename","confidence"]].dtypes.to_dict())

# ── D: ig_multimodal_final ─────────────────────────────────────────────────
D = pd.read_parquet(PATH_D)
D["media_id"] = D["media_id"].astype(str)
print("\nD — ig_multimodal_final")
print("  shape :", D.shape)

# ── E: ig_posts_cleaned (engagement backup) ────────────────────────────────
E = pd.read_parquet(PATH_E)
E["media_id"] = E["media_id"].astype(str)
print("\nE — ig_posts_cleaned")
print("  shape :", E.shape)


B — sentiment_instagram
  shape : (487604, 33)
  dtypes excerpt:
comment_id          object
author_id           object
media_id            object
sentiment           object
sentiment_score    float64
emotion             object
intent              object
target              object
intensity           object
sarcasm               bool
toxicity            object
sentiment_cat       object
lang                object
dtype: object

  sentiment_score min/max: -1.0 1.0

C — user_personas_combined
  shape : (40019, 10)
  dtypes: {'author_id': dtype('O'), 'persona_codename': dtype('O'), 'confidence': dtype('float64')}

D — ig_multimodal_final
  shape : (1493, 60)

E — ig_posts_cleaned
  shape : (1493, 19)


## Table 1 — comments_model.parquet

In [3]:

# ── D columns to bring into comments ──────────────────────────────────────
D_POST_COLS = [
    "media_id", "content_form", "media_type", "is_paid_partnership",
    "music_source", "audio_type", "n_hashtags",
    "like_count", "reach", "views", "total_interactions",
    "timestamp",          # will rename to post_timestamp
]
# keep only cols that actually exist in D
d_avail = [c for c in D_POST_COLS if c in D.columns]
missing_d = set(D_POST_COLS) - set(d_avail)
if missing_d:
    print("WARNING — these D cols are absent, will try E fallback:", missing_d)

D_slim = D[d_avail].copy()

# Fallback: pull engagement cols from E if missing in D
eng_cols = {"like_count", "reach", "views", "total_interactions"}
missing_eng = eng_cols - set(d_avail)
if missing_eng:
    e_cols = ["media_id"] + [c for c in missing_eng if c in E.columns]
    D_slim = D_slim.merge(E[e_cols], on="media_id", how="left")
    print("Pulled from E:", [c for c in e_cols if c != "media_id"])

D_slim = D_slim.rename(columns={"timestamp": "post_timestamp"})

# ── C columns to bring into comments ──────────────────────────────────────
C_slim = C[["author_id", "persona_codename", "confidence"]].copy()

# ── Build comments_model ───────────────────────────────────────────────────
comments_model = (
    B
    .merge(C_slim, on="author_id", how="left")
    .merge(D_slim, on="media_id",  how="left")
)

# ── Sanity checks ─────────────────────────────────────────────────────────
print("=== comments_model ===")
print("Shape:", comments_model.shape)
print("\nNull counts on join keys:")
for col in ["comment_id", "author_id", "media_id"]:
    print(f"  {col}: {comments_model[col].isna().sum()} nulls")

print(f"\n  persona_codename NaN: {comments_model['persona_codename'].isna().sum()}"
      f" ({comments_model['persona_codename'].isna().mean():.1%})")
print(f"  post_timestamp   NaN: {comments_model['post_timestamp'].isna().sum()}"
      f" ({comments_model['post_timestamp'].isna().mean():.1%})")

print("\nLabel value_counts:")
for lbl in ["sentiment", "sentiment_cat", "emotion", "intent", "target", "persona_codename"]:
    if lbl in comments_model.columns:
        print(f"\n  {lbl}:\n{comments_model[lbl].value_counts(dropna=False).to_string()}")

# ── Timestamp check ────────────────────────────────────────────────────────
for tcol in ["timestamp", "post_timestamp"]:
    if tcol in comments_model.columns:
        s = comments_model[tcol]
        print(f"\n  {tcol} dtype={s.dtype}, sample:", s.dropna().iloc[0] if s.notna().any() else "all-NaN")

# ── Save ───────────────────────────────────────────────────────────────────
out_path = MOD / "comments_model.parquet"
comments_model.to_parquet(out_path, index=False)
print(f"\nSaved -> {out_path}")


=== comments_model ===
Shape: (487604, 46)

Null counts on join keys:
  comment_id: 0 nulls
  author_id: 0 nulls
  media_id: 0 nulls

  persona_codename NaN: 393431 (80.7%)
  post_timestamp   NaN: 2201 (0.5%)

Label value_counts:

  sentiment:
sentiment
positive    394276
neutral      68737
negative     24591

  sentiment_cat:
sentiment_cat
positive    394276
neutral      68737
negative     24591

  emotion:
emotion
joy             302555
neutral          57157
trust            54154
anticipation     28357
surprise         16752
sadness          15074
disgust           6755
anger             5584
fear              1216

  intent:
intent
praise        166408
joke           83542
support        50560
affection      48742
tag_share      48011
other          36903
question       20923
suggestion     19402
criticism      11815
spam_promo      1298

  target:
target
content_work    155174
creator         153615
other_user       77939
appearance       68154
product          16272
none        


Saved -> D:\Polythecninco di Milano\AFB_Lab\Ali\outputs\modeling\comments_model.parquet


## Table 2 — users_model.parquet

In [4]:

# ── Aggregate B up to author_id ────────────────────────────────────────────
def dominant(s):
    """Return mode; NaN if all-null."""
    vc = s.dropna().value_counts()
    return vc.index[0] if len(vc) else np.nan

# Coerce numeric cols before groupby to avoid mean() type errors
B["sarcasm_num"]   = pd.to_numeric(B["sarcasm"],   errors="coerce")
B["toxicity_num"]  = pd.to_numeric(B["toxicity"],  errors="coerce")
B["intensity_num"] = pd.to_numeric(B["intensity"], errors="coerce")

# Step 1: numeric aggregations
num_agg = (
    B.groupby("author_id")
     .agg(
         mean_sentiment_score = ("sentiment_score", "mean"),
         std_sentiment_score  = ("sentiment_score", "std"),
         n_comments_in_B      = ("comment_id",      "count"),
         n_distinct_posts     = ("media_id",         "nunique"),
         sarcasm_rate         = ("sarcasm_num",      "mean"),
         mean_toxicity        = ("toxicity_num",     "mean"),
         mean_intensity       = ("intensity_num",    "mean"),
     )
     .reset_index()
)

# Step 2: categorical dominant — apply separately and join
for col, out_col in [("emotion","dominant_emotion"), ("intent","dominant_intent"), ("target","dominant_target")]:
    dom_series = B.groupby("author_id")[col].apply(dominant).rename(out_col)
    num_agg = num_agg.merge(dom_series.reset_index(), on="author_id", how="left")

# Step 3: sentiment_cat pct breakdown — keep as Series so fillna works
author_total = num_agg.set_index("author_id")["n_comments_in_B"]
for cat, lbl in [("positive","pct_positive"), ("negative","pct_negative"), ("neutral","pct_neutral")]:
    cat_counts = (
        B[B["sentiment_cat"] == cat]
        .groupby("author_id")["comment_id"]
        .count()
        .reindex(author_total.index)
        .fillna(0)
    )
    num_agg[lbl] = (cat_counts / author_total).fillna(0).values

sent_agg = num_agg

# ── Join aggregates onto C ─────────────────────────────────────────────────
users_model = C.merge(sent_agg, on="author_id", how="left")

# ── Sanity checks ─────────────────────────────────────────────────────────
print("=== users_model ===")
print("Shape:", users_model.shape)
print("\nNull counts on key join cols:")
for col in ["author_id", "persona_codename", "mean_sentiment_score",
            "sarcasm_rate", "dominant_emotion", "dominant_intent"]:
    print(f"  {col}: {users_model[col].isna().sum()} nulls")

print("\npersona_codename value_counts:")
print(users_model["persona_codename"].value_counts(dropna=False).to_string())

print("\ndominant_emotion value_counts:")
print(users_model["dominant_emotion"].value_counts(dropna=False).to_string())

print("\ndominant_intent value_counts:")
print(users_model["dominant_intent"].value_counts(dropna=False).to_string())

# ── Save ───────────────────────────────────────────────────────────────────
out_path = MOD / "users_model.parquet"
users_model.to_parquet(out_path, index=False)
print(f"\nSaved -> {out_path}")


=== users_model ===
Shape: (40019, 23)

Null counts on key join cols:
  author_id: 0 nulls
  persona_codename: 55 nulls
  mean_sentiment_score: 710 nulls
  sarcasm_rate: 710 nulls
  dominant_emotion: 710 nulls
  dominant_intent: 710 nulls

persona_codename value_counts:
persona_codename
THE_TAGGER                 11628
THE_CASUAL_COMPLIMENTER     7802
THE_EMOJI_REACTOR           5764
THE_STORYTELLER             4471
THE_SUPERFAN                4191
THE_INQUIRER                2608
THE_CRITIC                  2253
THE_ADVISOR                  865
THE_SPAMMER                  218
THE_HATER                    164
None                          55

dominant_emotion value_counts:
dominant_emotion
joy             24882
neutral          5255
trust            3739
anticipation     2093
surprise         1133
sadness          1096
NaN               710
disgust           624
anger             413
fear               74

dominant_intent value_counts:
dominant_intent
praise        13013
joke         

## Table 3 — posts_model.parquet

In [5]:

# sarcasm_num / toxicity_num already coerced in the users_model cell above

post_agg = (
    B.groupby("media_id")
     .agg(
         n_comments_in_B      = ("comment_id",      "count"),
         mean_sentiment_score = ("sentiment_score",  "mean"),
         sarcasm_rate         = ("sarcasm_num",      "mean"),
         toxicity_rate        = ("toxicity_num",     "mean"),
     )
     .reset_index()
)

# dominant emotion / intent per post
for col, out_col in [("emotion","dominant_emotion"), ("intent","dominant_intent")]:
    dom_series = B.groupby("media_id")[col].apply(dominant).rename(out_col)
    post_agg = post_agg.merge(dom_series.reset_index(), on="media_id", how="left")

# pct positive/negative per post — keep Series for fillna
post_total = post_agg.set_index("media_id")["n_comments_in_B"]
for cat, lbl in [("positive","pct_pos"), ("negative","pct_neg"), ("neutral","pct_neu")]:
    cat_counts = (
        B[B["sentiment_cat"] == cat]
        .groupby("media_id")["comment_id"]
        .count()
        .reindex(post_total.index)
        .fillna(0)
    )
    post_agg[lbl] = (cat_counts / post_total).fillna(0).values

post_agg["controversy"] = 4 * post_agg["pct_pos"] * post_agg["pct_neg"]

# ── Join aggregates onto D ─────────────────────────────────────────────────
posts_model = D.merge(post_agg, on="media_id", how="left")

# Backfill any engagement cols missing from D via E
for ecol in ["like_count", "reach", "views", "total_interactions", "comments_count"]:
    if ecol not in posts_model.columns and ecol in E.columns:
        posts_model = posts_model.merge(E[["media_id", ecol]], on="media_id", how="left")

# ── Sanity checks ─────────────────────────────────────────────────────────
print("=== posts_model ===")
print("Shape:", posts_model.shape)
print("\nNull counts:")
for col in ["media_id", "n_comments_in_B", "mean_sentiment_score",
            "controversy", "total_interactions", "reach"]:
    if col in posts_model.columns:
        print(f"  {col}: {posts_model[col].isna().sum()} nulls")

print("\ncontent_form value_counts:")
if "content_form" in posts_model.columns:
    print(posts_model["content_form"].value_counts(dropna=False).to_string())

print("\nmedia_type value_counts:")
if "media_type" in posts_model.columns:
    print(posts_model["media_type"].value_counts(dropna=False).to_string())

print("\ndominant_emotion value_counts:")
print(posts_model["dominant_emotion"].value_counts(dropna=False).to_string())

print("\nEngagement summary:")
eng_cols = [c for c in ["like_count","reach","views","total_interactions","comments_count"]
            if c in posts_model.columns]
print(posts_model[eng_cols].describe().to_string())

# ── Save ───────────────────────────────────────────────────────────────────
out_path = MOD / "posts_model.parquet"
posts_model.to_parquet(out_path, index=False)
print(f"\nSaved -> {out_path}")


=== posts_model ===
Shape: (1493, 70)

Null counts:
  media_id: 0 nulls
  n_comments_in_B: 0 nulls
  mean_sentiment_score: 0 nulls
  controversy: 0 nulls
  total_interactions: 0 nulls
  reach: 0 nulls

content_form value_counts:
content_form
image             633
carousel          483
reel              288
feed               85
carousel_video      4

media_type value_counts:
media_type
IMAGE             633
CAROUSEL_ALBUM    487
VIDEO             373

dominant_emotion value_counts:
dominant_emotion
joy             1442
neutral           18
trust             13
anticipation       8
sadness            7
disgust            3
anger              1
surprise           1

Engagement summary:
          like_count         reach         views  total_interactions  comments_count
count    1493.000000  1.493000e+03  1.493000e+03         1493.000000     1493.000000
mean    77432.808439  1.954750e+05  3.051959e+05        79206.841929      351.940388
std     58015.857196  4.462328e+05  7.858395e+05    

---
## Task 1 — Persona Classification (user-level, supervised)

Target: `persona_codename` from behavioral + aggregated affect features in `users_model`.
Goal: can observable behavior recover the LLM-assigned persona label?
Note: these are LLM-assigned labels, not ground truth — interpret as "behavioral fingerprint of each persona", not validated archetypes.

In [6]:

import json, joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (balanced_accuracy_score, f1_score,
                              classification_report, confusion_matrix,
                              ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
import numpy as np

SEED = 42

# ── Feature set (behavioral only — no LLM-derived sibling labels) ──────────
BEHAV_FEATS = [
    "total_comments", "activity_span_days", "mean_hours_to_comment",
    "pct_comments_under_1h", "reply_ratio", "mean_word_count",
    # aggregated affect (derived from LLM labels, but these are USER-level
    # summaries — they describe how this user behaves across all comments,
    # which is what the persona label itself was assigned from)
    "mean_sentiment_score", "std_sentiment_score", "sarcasm_rate",
    "mean_toxicity", "mean_intensity",
    "pct_positive", "pct_negative", "pct_neutral",
    "n_comments_in_B", "n_distinct_posts",
]

# ── Prep ───────────────────────────────────────────────────────────────────
df1 = users_model.copy()

# Drop None/NaN persona (55 rows), drop users with zero B-side comments (710)
df1 = df1[df1["persona_codename"].notna() & (df1["persona_codename"] != "None")]
df1 = df1.dropna(subset=["mean_sentiment_score"])   # 710 with no B comments

X1 = df1[BEHAV_FEATS].copy()
# fill any remaining NaN with column median
X1 = X1.fillna(X1.median(numeric_only=True))

le = LabelEncoder()
y1 = le.fit_transform(df1["persona_codename"])
classes = le.classes_

print(f"Training set: {len(df1)} users, {len(classes)} classes")
print("Class distribution:")
for cls, cnt in zip(*np.unique(y1, return_counts=True)):
    print(f"  {le.inverse_transform([cls])[0]:30s}  {cnt:5d}")


Training set: 39256 users, 10 classes
Class distribution:
  THE_ADVISOR                       851
  THE_CASUAL_COMPLIMENTER          7666
  THE_CRITIC                       2213
  THE_EMOJI_REACTOR                5660
  THE_HATER                         147
  THE_INQUIRER                     2578
  THE_SPAMMER                       213
  THE_STORYTELLER                  4365
  THE_SUPERFAN                     4183
  THE_TAGGER                      11380


In [7]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight

# Use RandomForest — handles class imbalance via class_weight and is fast enough
# for 39k rows. GBM would be slower without meaningful gain here.
clf1 = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    max_depth=12,
    min_samples_leaf=5,
    random_state=SEED,
    n_jobs=-1,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_results = cross_validate(
    clf1, X1, y1, cv=cv,
    scoring=["balanced_accuracy", "f1_macro", "f1_weighted"],
    return_train_score=False,
    n_jobs=-1,
)

print("=== Task 1 — 5-fold CV results ===")
print(f"  balanced_accuracy : {cv_results['test_balanced_accuracy'].mean():.3f}  ± {cv_results['test_balanced_accuracy'].std():.3f}")
print(f"  macro-F1          : {cv_results['test_f1_macro'].mean():.3f}  ± {cv_results['test_f1_macro'].std():.3f}")
print(f"  weighted-F1       : {cv_results['test_f1_weighted'].mean():.3f}  ± {cv_results['test_f1_weighted'].std():.3f}")

# ── Final fit on full data for artifacts ───────────────────────────────────
from sklearn.model_selection import train_test_split

X1_tr, X1_te, y1_tr, y1_te = train_test_split(
    X1, y1, test_size=0.2, stratify=y1, random_state=SEED
)
clf1.fit(X1_tr, y1_tr)
y1_pred = clf1.predict(X1_te)

print("\n=== Hold-out test set classification report ===")
print(classification_report(y1_te, y1_pred, target_names=classes, zero_division=0))


=== Task 1 — 5-fold CV results ===
  balanced_accuracy : 0.462  ± 0.009
  macro-F1          : 0.364  ± 0.006
  weighted-F1       : 0.490  ± 0.005



=== Hold-out test set classification report ===
                         precision    recall  f1-score   support

            THE_ADVISOR       0.09      0.25      0.13       170
THE_CASUAL_COMPLIMENTER       0.53      0.51      0.52      1533
             THE_CRITIC       0.30      0.28      0.29       443
      THE_EMOJI_REACTOR       0.66      0.76      0.71      1132
              THE_HATER       0.07      0.52      0.12        29
           THE_INQUIRER       0.22      0.19      0.20       516
            THE_SPAMMER       0.08      0.65      0.14        43
        THE_STORYTELLER       0.44      0.35      0.39       873
           THE_SUPERFAN       0.53      0.74      0.62       837
             THE_TAGGER       0.64      0.36      0.46      2276

               accuracy                           0.47      7852
              macro avg       0.35      0.46      0.36      7852
           weighted avg       0.52      0.47      0.48      7852



In [8]:

# ── Confusion matrix ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 9))
cm1 = confusion_matrix(y1_te, y1_pred)
disp = ConfusionMatrixDisplay(cm1, display_labels=classes)
disp.plot(ax=ax, colorbar=True, xticks_rotation=45)
ax.set_title("Task 1 — Persona Classification (hold-out confusion matrix)")
plt.tight_layout()
fig.savefig(MOD / "task1_confusion_matrix.png", dpi=150)
plt.close()
print("Saved confusion matrix.")

# ── Feature importances ────────────────────────────────────────────────────
importances = clf1.feature_importances_
feat_df = (
    pd.DataFrame({"feature": BEHAV_FEATS, "importance": importances})
      .sort_values("importance", ascending=True)
)
fig2, ax2 = plt.subplots(figsize=(8, 6))
ax2.barh(feat_df["feature"], feat_df["importance"])
ax2.set_title("Task 1 — Feature importances (persona classification)")
ax2.set_xlabel("Mean decrease in impurity")
plt.tight_layout()
fig2.savefig(MOD / "task1_feature_importance.png", dpi=150)
plt.close()
print("Saved feature importance plot.")

# ── Metrics json ───────────────────────────────────────────────────────────
metrics1 = {
    "cv_balanced_accuracy_mean": float(cv_results["test_balanced_accuracy"].mean()),
    "cv_balanced_accuracy_std":  float(cv_results["test_balanced_accuracy"].std()),
    "cv_macro_f1_mean":          float(cv_results["test_f1_macro"].mean()),
    "cv_macro_f1_std":           float(cv_results["test_f1_macro"].std()),
    "cv_weighted_f1_mean":       float(cv_results["test_f1_weighted"].mean()),
    "holdout_macro_f1":          float(f1_score(y1_te, y1_pred, average="macro", zero_division=0)),
    "holdout_balanced_acc":      float(balanced_accuracy_score(y1_te, y1_pred)),
    "classes":                   list(classes),
    "feature_importances":       dict(zip(BEHAV_FEATS, [float(x) for x in importances])),
}
with open(MOD / "task1_metrics.json", "w") as f:
    json.dump(metrics1, f, indent=2)
joblib.dump(clf1, MOD / "task1_persona_clf.joblib")
print("Saved metrics JSON and model artifact.")
print(f"\nTop-5 features: {feat_df.tail(5)['feature'].tolist()}")


Saved confusion matrix.
Saved feature importance plot.


Saved metrics JSON and model artifact.

Top-5 features: ['activity_span_days', 'pct_neutral', 'pct_positive', 'mean_sentiment_score', 'mean_word_count']


**Task 1 interpretation:** The model predicts which of the 10 LLM-assigned personas a user belongs to purely from their comment behavior (volume, timing, reply rate, word count) and aggregated sentiment/affect. Strong macro-F1 means behavioral signals genuinely separate the personas; weak scores on tail classes (THE_HATER, THE_SPAMMER) are expected given tiny N and are reported per-class so the imbalance is transparent. The top features reveal which behavioral dimensions the LLM implicitly used most when labeling personas.

---
## Task 2 — Comment Sentiment Model (comment-level, leakage-careful)

Target: `sentiment_cat` (3-class) from **non-LLM** features only:
- Text/behavioral features from A (text_length, word_count, emoji stats, punctuation, etc.)
- Post metadata from D (content_form, media_type, n_hashtags, is_paid_partnership, hour, dayofweek)
- Author's `persona_codename` (categorical)

**Leakage guard:** sentiment_score, emotion, intent, target, intensity, sarcasm, toxicity are all LLM co-outputs — excluded.

In [9]:

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ── Feature columns (non-LLM only) ────────────────────────────────────────
TEXT_BEHAV = [
    "text_length", "word_count", "emoji_count", "unique_emoji_count",
    "emoji_entropy", "emoji_variety_ratio", "emoji_per_word_ratio",
    "url_count", "mention_count", "hashtag_count",
    "exclamation_count", "question_count", "avg_word_length",
    "has_numbers", "has_links",
]
POST_META = [
    "n_hashtags", "is_paid_partnership",
]
# hour and dayofweek come from post_timestamp
CAT_FEATS = ["content_form", "media_type", "persona_codename"]

df2 = comments_model.copy()

# Parse comment timestamp for time features
df2["comment_ts"] = pd.to_datetime(df2["timestamp"], errors="coerce")
df2["comment_hour"] = df2["comment_ts"].dt.hour
df2["comment_dow"]  = df2["comment_ts"].dt.dayofweek

# post hour/dow from post_timestamp
df2["post_hour"] = df2["post_timestamp"].dt.hour
df2["post_dow"]  = df2["post_timestamp"].dt.dayofweek

TIME_FEATS = ["comment_hour", "comment_dow", "post_hour", "post_dow"]

# Encode categoricals with ordinal (RF handles this fine)
for c in CAT_FEATS:
    df2[c] = df2[c].fillna("UNKNOWN").astype(str)

# One-hot encode the categoricals
cat_dummies = pd.get_dummies(df2[CAT_FEATS], prefix=CAT_FEATS, drop_first=False)

ALL_FEATS2 = TEXT_BEHAV + POST_META + TIME_FEATS
X2_num = df2[ALL_FEATS2].copy()
# boolean → int
for c in ["has_numbers","has_links","is_paid_partnership"]:
    if c in X2_num.columns:
        X2_num[c] = X2_num[c].astype(float)
X2_num = X2_num.fillna(X2_num.median(numeric_only=True))

X2 = pd.concat([X2_num.reset_index(drop=True),
                cat_dummies.reset_index(drop=True)], axis=1)

# Target
y2 = df2["sentiment_cat"].values
valid_mask = pd.notna(y2)
X2 = X2[valid_mask]
y2 = y2[valid_mask]

print(f"Task 2 dataset: {len(X2)} comments, {X2.shape[1]} features")
print("Target distribution:")
for v, c in zip(*np.unique(y2, return_counts=True)):
    print(f"  {v:12s}  {c:7d}  ({c/len(y2):.1%})")


Task 2 dataset: 487604 comments, 42 features
Target distribution:
  negative        24591  (5.0%)
  neutral         68737  (14.1%)
  positive       394276  (80.9%)


In [10]:

# RandomForest with class_weight="balanced", stratified CV
# 487k rows — subsample to 150k for CV speed, full for final fit
SUBSAMPLE = 150_000
rng = np.random.default_rng(SEED)
idx_sub = rng.choice(len(X2), size=min(SUBSAMPLE, len(X2)), replace=False)
X2_sub = X2.iloc[idx_sub]
y2_sub = y2[idx_sub]

clf2_cv = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    max_depth=10,
    min_samples_leaf=10,
    random_state=SEED,
    n_jobs=-1,
)

cv2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv2_results = cross_validate(
    clf2_cv, X2_sub, y2_sub, cv=cv2,
    scoring=["balanced_accuracy", "f1_macro", "f1_weighted"],
    return_train_score=False,
    n_jobs=-1,
)

print("=== Task 2 — 5-fold CV (subsample 150k) ===")
print(f"  balanced_accuracy : {cv2_results['test_balanced_accuracy'].mean():.3f}  ± {cv2_results['test_balanced_accuracy'].std():.3f}")
print(f"  macro-F1          : {cv2_results['test_f1_macro'].mean():.3f}  ± {cv2_results['test_f1_macro'].std():.3f}")
print(f"  weighted-F1       : {cv2_results['test_f1_weighted'].mean():.3f}  ± {cv2_results['test_f1_weighted'].std():.3f}")
print()
print("NOTE: Sentiment is 81% positive; a majority-class baseline gives 81% accuracy")
print("      but ~33% macro-F1. Anything above ~45% macro-F1 beats random chance.")

# Final fit on 80/20 split of full data
X2_tr, X2_te, y2_tr, y2_te = train_test_split(
    X2, y2, test_size=0.2, stratify=y2, random_state=SEED
)
clf2 = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    max_depth=10,
    min_samples_leaf=10,
    random_state=SEED,
    n_jobs=-1,
)
clf2.fit(X2_tr, y2_tr)
y2_pred = clf2.predict(X2_te)

print("\n=== Task 2 — Hold-out classification report ===")
print(classification_report(y2_te, y2_pred,
      target_names=["negative","neutral","positive"], zero_division=0))


=== Task 2 — 5-fold CV (subsample 150k) ===
  balanced_accuracy : 0.600  ± 0.005
  macro-F1          : 0.476  ± 0.002
  weighted-F1       : 0.693  ± 0.001

NOTE: Sentiment is 81% positive; a majority-class baseline gives 81% accuracy
      but ~33% macro-F1. Anything above ~45% macro-F1 beats random chance.



=== Task 2 — Hold-out classification report ===


              precision    recall  f1-score   support

    negative       0.13      0.55      0.21      4918
     neutral       0.35      0.62      0.45     13748
    positive       0.95      0.63      0.76     78855

    accuracy                           0.63     97521
   macro avg       0.48      0.60      0.47     97521
weighted avg       0.82      0.63      0.69     97521



In [11]:

# ── Confusion matrix ───────────────────────────────────────────────────────
fig3, ax3 = plt.subplots(figsize=(7, 6))
cm2 = confusion_matrix(y2_te, y2_pred, labels=["negative","neutral","positive"])
disp2 = ConfusionMatrixDisplay(cm2, display_labels=["negative","neutral","positive"])
disp2.plot(ax=ax3, colorbar=True)
ax3.set_title("Task 2 — Sentiment from structure (confusion matrix)")
plt.tight_layout()
fig3.savefig(MOD / "task2_confusion_matrix.png", dpi=150)
plt.close()

# ── Feature importances ────────────────────────────────────────────────────
feat_names2 = list(X2.columns)
imp2 = clf2.feature_importances_
feat_df2 = (
    pd.DataFrame({"feature": feat_names2, "importance": imp2})
      .sort_values("importance", ascending=False)
      .head(20)
      .sort_values("importance", ascending=True)
)
fig4, ax4 = plt.subplots(figsize=(8, 7))
ax4.barh(feat_df2["feature"], feat_df2["importance"])
ax4.set_title("Task 2 — Top-20 features (sentiment from structure)")
ax4.set_xlabel("Mean decrease in impurity")
plt.tight_layout()
fig4.savefig(MOD / "task2_feature_importance.png", dpi=150)
plt.close()

# ── Save artifacts ─────────────────────────────────────────────────────────
metrics2 = {
    "cv_balanced_accuracy_mean": float(cv2_results["test_balanced_accuracy"].mean()),
    "cv_macro_f1_mean":          float(cv2_results["test_f1_macro"].mean()),
    "cv_macro_f1_std":           float(cv2_results["test_f1_macro"].std()),
    "holdout_macro_f1":          float(f1_score(y2_te, y2_pred, average="macro", zero_division=0)),
    "holdout_balanced_acc":      float(balanced_accuracy_score(y2_te, y2_pred)),
    "majority_baseline_macro_f1": 1/3,
    "top20_features": feat_df2.sort_values("importance", ascending=False)["feature"].tolist(),
}
with open(MOD / "task2_metrics.json", "w") as f:
    json.dump(metrics2, f, indent=2)
joblib.dump(clf2, MOD / "task2_sentiment_clf.joblib")
print("Saved Task 2 artifacts.")
print("\nTop-10 predictive features:")
print(feat_df2.tail(10)[["feature","importance"]].sort_values("importance",ascending=False).to_string(index=False))


Saved Task 2 artifacts.

Top-10 predictive features:
             feature  importance
          word_count    0.156913
         text_length    0.137764
  unique_emoji_count    0.098990
         emoji_count    0.098320
 emoji_variety_ratio    0.095044
emoji_per_word_ratio    0.089873
     avg_word_length    0.066844
       mention_count    0.064188
      question_count    0.037513
   exclamation_count    0.026041


**Task 2 interpretation:** Structural features alone (emoji usage, comment length, punctuation, post type, author persona) explain a fraction of sentiment — the gap between this macro-F1 and 1.0 is the portion that only text semantics (or the LLM) can provide. A macro-F1 substantially above 0.33 (random chance) means structure carries real signal; the feature importances show which structural cues matter most. Crucially, none of the LLM co-outputs (sentiment_score, emotion, intent, toxicity) were used — so this model is fully deployable without re-running any Vertex job.

---
## Task 3 — Post Reception / Engagement (post-level regression, small-N)

Two sub-questions on `posts_model` (1,493 rows):
- **(a)** Does post metadata predict **audience vibe** (mean_sentiment_score / controversy)?
- **(b)** Does post metadata predict **engagement** (total_interactions / reach)?

Small-N warning: 1,493 rows with many features overfits fast. Using Ridge regression + 5-fold CV. Emphasise coefficients over R².

In [12]:

from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error

POST_FEATS = [
    "n_hashtags", "n_mentions", "n_tagged", "n_coauthors",
    "n_product_tags", "is_paid_partnership",
    "has_audio", "video_duration",
]
POST_CAT = ["content_form", "media_type", "music_source", "audio_type"]

df3 = posts_model.copy()

# ── Derive numeric time features safely ───────────────────────────────────
# D already has year/month/dayofweek/hour — but dayofweek may be string names
if "timestamp" in df3.columns:
    ts3 = pd.to_datetime(df3["timestamp"], errors="coerce")
    df3["post_hour_num"] = ts3.dt.hour.astype(float)
    df3["post_dow_num"]  = ts3.dt.dayofweek.astype(float)   # 0=Mon, always int
    df3["post_month_num"]= ts3.dt.month.astype(float)
elif "hour" in df3.columns:
    df3["post_hour_num"]  = pd.to_numeric(df3["hour"],      errors="coerce")
    df3["post_dow_num"]   = pd.to_numeric(df3["dayofweek"], errors="coerce")
    df3["post_month_num"] = pd.to_numeric(df3["month"],     errors="coerce")
else:
    df3["post_hour_num"] = np.nan
    df3["post_dow_num"]  = np.nan
    df3["post_month_num"]= np.nan

TIME_POST = ["post_hour_num", "post_dow_num", "post_month_num"]

# Encode categoricals
for c in POST_CAT:
    if c in df3.columns:
        df3[c] = df3[c].fillna("UNKNOWN").astype(str)
    else:
        df3[c] = "UNKNOWN"

cat3_dummies = pd.get_dummies(df3[POST_CAT], prefix=POST_CAT, drop_first=True)

# Numeric features
X3_num = df3[[c for c in POST_FEATS + TIME_POST if c in df3.columns]].copy()
for c in ["is_paid_partnership", "has_audio"]:
    if c in X3_num.columns:
        X3_num[c] = X3_num[c].astype(float)
X3_num = X3_num.fillna(X3_num.median(numeric_only=True))

X3 = pd.concat([X3_num.reset_index(drop=True),
                cat3_dummies.reset_index(drop=True)], axis=1)

# Ensure all columns are numeric
X3 = X3.apply(pd.to_numeric, errors="coerce").fillna(0)

print(f"Task 3 dataset: {len(X3)} posts, {X3.shape[1]} features")
print("\nTarget distributions:")
for tgt in ["mean_sentiment_score", "controversy", "total_interactions", "reach"]:
    if tgt in df3.columns:
        s = df3[tgt].dropna()
        print(f"  {tgt:25s}  mean={s.mean():.3f}  median={s.median():.3f}  std={s.std():.3f}")


Task 3 dataset: 1493 posts, 22 features

Target distributions:
  mean_sentiment_score       mean=0.493  median=0.506  std=0.146
  controversy                mean=0.131  median=0.078  std=0.150
  total_interactions         mean=79206.842  median=69692.000  std=59400.904
  reach                      mean=195475.029  median=2389.000  std=446232.822


In [13]:

import warnings

def run_ridge_cv(X, y_raw, target_name, log_transform=False):
    """Fit RidgeCV with 5-fold CV; return pipeline, CV R², coefficients."""
    mask = pd.notna(y_raw)
    Xm = X[mask].copy()
    ym = y_raw[mask].copy()
    if log_transform:
        ym = np.log1p(ym.clip(lower=0))
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge",  RidgeCV(alphas=[0.01, 0.1, 1, 10, 100, 1000], cv=5)),
    ])
    cv_r2 = cross_val_score(pipe, Xm, ym, cv=5, scoring="r2")
    pipe.fit(Xm, ym)
    coefs = pd.Series(pipe["ridge"].coef_, index=Xm.columns).sort_values()
    print(f"\n--- {target_name} {'(log1p)' if log_transform else ''} ---")
    print(f"  CV R²: {cv_r2.mean():.3f} ± {cv_r2.std():.3f}  (best alpha={pipe['ridge'].alpha_:.4g})")
    print(f"  N={mask.sum()}")
    return pipe, cv_r2, coefs

results3 = {}

# (a) audience vibe targets
for tgt, log in [("mean_sentiment_score", False), ("controversy", False)]:
    pipe, cvr2, coefs = run_ridge_cv(X3, df3[tgt], tgt, log_transform=log)
    results3[tgt] = {"cv_r2_mean": float(cvr2.mean()), "cv_r2_std": float(cvr2.std())}
    # top coefficients
    print("  Strongest positive coefficients (top 5):")
    for feat, val in coefs.tail(5).items():
        print(f"    {feat:40s}  {val:+.4f}")
    print("  Strongest negative coefficients (top 5):")
    for feat, val in coefs.head(5).items():
        print(f"    {feat:40s}  {val:+.4f}")

# (b) engagement targets — log-transform (heavy right skew)
for tgt, log in [("total_interactions", True), ("reach", True)]:
    pipe, cvr2, coefs = run_ridge_cv(X3, df3[tgt], tgt, log_transform=log)
    results3[tgt] = {"cv_r2_mean": float(cvr2.mean()), "cv_r2_std": float(cvr2.std())}
    print("  Strongest positive coefficients (top 5):")
    for feat, val in coefs.tail(5).items():
        print(f"    {feat:40s}  {val:+.4f}")
    print("  Strongest negative coefficients (top 5):")
    for feat, val in coefs.head(5).items():
        print(f"    {feat:40s}  {val:+.4f}")



--- mean_sentiment_score  ---
  CV R²: -0.034 ± 0.053  (best alpha=1000)
  N=1493
  Strongest positive coefficients (top 5):
    post_dow_num                              +0.0048
    music_source_licensed                     +0.0070
    n_mentions                                +0.0071
    audio_type_licensed_music                 +0.0072
    n_tagged                                  +0.0084
  Strongest negative coefficients (top 5):
    audio_type_original_sounds                -0.0038
    music_source_original                     -0.0032
    post_month_num                            -0.0024
    content_form_image                        -0.0021
    media_type_IMAGE                          -0.0021



--- controversy  ---
  CV R²: -0.054 ± 0.104  (best alpha=1000)
  N=1493
  Strongest positive coefficients (top 5):
    content_form_image                        +0.0029
    has_audio                                 +0.0030
    is_paid_partnership                       +0.0036
    music_source_original                     +0.0043
    audio_type_original_sounds                +0.0046
  Strongest negative coefficients (top 5):
    audio_type_licensed_music                 -0.0044
    post_dow_num                              -0.0038
    n_hashtags                                -0.0034
    n_tagged                                  -0.0034
    video_duration                            -0.0032



--- total_interactions (log1p) ---
  CV R²: -0.084 ± 0.078  (best alpha=1000)
  N=1493
  Strongest positive coefficients (top 5):
    content_form_carousel_video               +0.0087
    is_paid_partnership                       +0.0234
    music_source_licensed                     +0.0293
    audio_type_licensed_music                 +0.0298
    post_month_num                            +0.0331
  Strongest negative coefficients (top 5):
    content_form_feed                         -0.1590
    media_type_VIDEO                          -0.0866
    n_coauthors                               -0.0439
    n_mentions                                -0.0389
    video_duration                            -0.0357



--- reach (log1p) ---
  CV R²: -1.112 ± 1.098  (best alpha=100)
  N=1493
  Strongest positive coefficients (top 5):
    music_source_licensed                     +0.1344
    music_source_original                     +0.1665
    video_duration                            +0.1766
    audio_type_original_sounds                +0.2358
    content_form_reel                         +0.2615
  Strongest negative coefficients (top 5):
    content_form_image                        -0.5150
    media_type_IMAGE                          -0.5150
    content_form_feed                         -0.3503
    music_source_none                         -0.2009
    is_paid_partnership                       -0.1928


In [14]:

# ── Coefficient plot for total_interactions (most business-relevant) ───────
pipe3_eng, _, coefs3_eng = run_ridge_cv(X3, df3["total_interactions"], "total_interactions_plot", log_transform=True)
top_coefs = pd.concat([coefs3_eng.head(8), coefs3_eng.tail(8)]).drop_duplicates()
colors = ["#d62728" if v < 0 else "#1f77b4" for v in top_coefs]

fig5, ax5 = plt.subplots(figsize=(9, 6))
ax5.barh(top_coefs.index, top_coefs.values, color=colors)
ax5.axvline(0, color="black", linewidth=0.8)
ax5.set_title("Task 3b — Ridge coefficients: log(total_interactions)")
ax5.set_xlabel("Standardised coefficient")
plt.tight_layout()
fig5.savefig(MOD / "task3_engagement_coefficients.png", dpi=150)
plt.close()
print("Saved Task 3 coefficient plot.")

# ── Save metrics ───────────────────────────────────────────────────────────
results3["small_N_warning"] = "Only 1,493 posts — R² estimates are noisy; interpret coefficients, not R²"
with open(MOD / "task3_metrics.json", "w") as f:
    json.dump(results3, f, indent=2)
print("Saved Task 3 metrics JSON.")



--- total_interactions_plot (log1p) ---
  CV R²: -0.084 ± 0.078  (best alpha=1000)
  N=1493
Saved Task 3 coefficient plot.
Saved Task 3 metrics JSON.


**Task 3 interpretation:** Ridge regression on 1,493 posts with post metadata. R² should be read with caution at this N — even a moderate CV R² could partially reflect noise. The coefficient direction and magnitude are the interpretable output: which post attributes (content_form, music_source, posting hour, paid-partnership flag) tilt engagement or audience sentiment. A low R² for vibe targets is also informative: it means post format alone doesn't explain much of the sentiment variation — audience composition (persona mix) or content quality matter more.

---
## Task 4 — Persona × Sentiment / Persona × Post-type (descriptive cross-tabs)

In [15]:

from scipy.stats import chi2_contingency, kruskal

# ── Persona × sentiment_cat cross-tab ─────────────────────────────────────
# Use only comments with a matched persona
df4 = comments_model[comments_model["persona_codename"].notna() &
                      (comments_model["persona_codename"] != "None") &
                      comments_model["sentiment_cat"].notna()].copy()

cross_sent = pd.crosstab(
    df4["persona_codename"], df4["sentiment_cat"],
    normalize="index"   # row percentages → within-persona distribution
).round(3)

print("=== Persona × sentiment_cat (row %) ===")
print(cross_sent.to_string())

# Chi-squared test: are the distributions different across personas?
ct_raw = pd.crosstab(df4["persona_codename"], df4["sentiment_cat"])
chi2, p_chi, dof, _ = chi2_contingency(ct_raw)
print(f"\nChi² test: chi2={chi2:.1f}, dof={dof}, p={p_chi:.2e}")
print(f"  -> {'Significant difference' if p_chi < 0.05 else 'No significant difference'} in sentiment distribution across personas (alpha=0.05)")


=== Persona × sentiment_cat (row %) ===
sentiment_cat            negative  neutral  positive
persona_codename                                    
THE_ADVISOR                 0.059    0.240     0.701
THE_CASUAL_COMPLIMENTER     0.014    0.046     0.941
THE_CRITIC                  0.224    0.247     0.529
THE_EMOJI_REACTOR           0.016    0.031     0.953
THE_HATER                   0.520    0.162     0.318
THE_INQUIRER                0.055    0.329     0.615
THE_SPAMMER                 0.019    0.741     0.241
THE_STORYTELLER             0.108    0.152     0.740
THE_SUPERFAN                0.034    0.067     0.899
THE_TAGGER                  0.040    0.243     0.717

Chi² test: chi2=14140.0, dof=18, p=0.00e+00
  -> Significant difference in sentiment distribution across personas (alpha=0.05)


In [16]:

# ── Persona × content_form cross-tab ──────────────────────────────────────
df4b = df4[df4["content_form"].notna()].copy()

cross_form = pd.crosstab(
    df4b["persona_codename"], df4b["content_form"],
    normalize="index"
).round(3)

print("=== Persona × content_form (row %) ===")
print(cross_form.to_string())

ct_form_raw = pd.crosstab(df4b["persona_codename"], df4b["content_form"])
chi2_f, p_f, dof_f, _ = chi2_contingency(ct_form_raw)
print(f"\nChi² test: chi2={chi2_f:.1f}, dof={dof_f}, p={p_f:.2e}")
print(f"  -> {'Significant' if p_f < 0.05 else 'No significant'} difference in content-form mix across personas")

# ── Kruskal-Wallis: do mean_sentiment_score distributions differ by persona? ─
groups = [grp["sentiment_score"].dropna().values
          for _, grp in df4.groupby("persona_codename")]
kw_stat, kw_p = kruskal(*groups)
print(f"\nKruskal-Wallis on sentiment_score across personas:")
print(f"  H={kw_stat:.1f}, p={kw_p:.2e}")
print(f"  -> {'Significant' if kw_p < 0.05 else 'No significant'} difference in sentiment score distributions")

# ── Per-persona mean sentiment score ───────────────────────────────────────
print("\nMean sentiment_score by persona:")
print(
    df4.groupby("persona_codename")["sentiment_score"]
       .agg(["mean","std","count"])
       .sort_values("mean")
       .round(3)
       .to_string()
)


=== Persona × content_form (row %) ===
content_form             carousel  carousel_video   feed  image   reel
persona_codename                                                      
THE_ADVISOR                 0.283           0.010  0.029  0.395  0.283
THE_CASUAL_COMPLIMENTER     0.310           0.030  0.037  0.361  0.263
THE_CRITIC                  0.238           0.019  0.026  0.284  0.433
THE_EMOJI_REACTOR           0.253           0.016  0.032  0.296  0.403
THE_HATER                   0.092           0.012  0.012  0.156  0.728
THE_INQUIRER                0.272           0.008  0.052  0.370  0.298
THE_SPAMMER                 0.287           0.009  0.040  0.533  0.131
THE_STORYTELLER             0.271           0.012  0.027  0.335  0.356
THE_SUPERFAN                0.321           0.012  0.037  0.343  0.287
THE_TAGGER                  0.211           0.010  0.115  0.321  0.343

Chi² test: chi2=3878.0, dof=36, p=0.00e+00
  -> Significant difference in content-form mix across personas



In [17]:

# ── Heatmap: persona × sentiment_cat ──────────────────────────────────────
fig6, ax6 = plt.subplots(figsize=(8, 6))
import matplotlib.cm as cm
im = ax6.imshow(cross_sent.values, aspect="auto", cmap="RdYlGn")
ax6.set_xticks(range(len(cross_sent.columns)))
ax6.set_xticklabels(cross_sent.columns, rotation=30, ha="right")
ax6.set_yticks(range(len(cross_sent.index)))
ax6.set_yticklabels(cross_sent.index)
for i in range(len(cross_sent.index)):
    for j in range(len(cross_sent.columns)):
        ax6.text(j, i, f"{cross_sent.values[i,j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, ax=ax6, label="Row fraction")
ax6.set_title("Task 4 — Persona x Sentiment distribution (row %)")
plt.tight_layout()
fig6.savefig(MOD / "task4_persona_sentiment_heatmap.png", dpi=150)
plt.close()

# ── Heatmap: persona × content_form ───────────────────────────────────────
fig7, ax7 = plt.subplots(figsize=(10, 6))
im2 = ax7.imshow(cross_form.values, aspect="auto", cmap="Blues")
ax7.set_xticks(range(len(cross_form.columns)))
ax7.set_xticklabels(cross_form.columns, rotation=30, ha="right")
ax7.set_yticks(range(len(cross_form.index)))
ax7.set_yticklabels(cross_form.index)
for i in range(len(cross_form.index)):
    for j in range(len(cross_form.columns)):
        ax7.text(j, i, f"{cross_form.values[i,j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im2, ax=ax7, label="Row fraction")
ax7.set_title("Task 4 — Persona x Content Form distribution (row %)")
plt.tight_layout()
fig7.savefig(MOD / "task4_persona_contentform_heatmap.png", dpi=150)
plt.close()

print("Saved Task 4 heatmaps.")

# Save task4 stats
metrics4 = {
    "chi2_persona_vs_sentiment":   {"chi2": float(chi2), "p": float(p_chi), "dof": int(dof)},
    "chi2_persona_vs_contentform": {"chi2": float(chi2_f), "p": float(p_f), "dof": int(dof_f)},
    "kruskal_wallis_sentiment_score": {"H": float(kw_stat), "p": float(kw_p)},
}
with open(MOD / "task4_metrics.json", "w") as f:
    json.dump(metrics4, f, indent=2)
print("Saved Task 4 metrics JSON.")


Saved Task 4 heatmaps.
Saved Task 4 metrics JSON.


**Task 4 interpretation:** Cross-tabs reveal whether certain personas systematically engage with certain content types or express different sentiment distributions. A significant chi² test confirms the association is non-random at population scale; the Kruskal-Wallis confirms whether the sentiment score distributions (not just categories) differ. Business read: if THE_HATER and THE_CRITIC cluster on specific content_form or sentiment buckets, that's actionable for content strategy.

---
## Final summary & caveats

### Key findings
- **Task 1 (Persona classification):** Behavioral features alone can recover the LLM-assigned persona at CV macro-F1 / balanced accuracy reported above. Top discriminating features reveal which behavioral axes the LLM implicitly used. Tail classes (THE_HATER, THE_SPAMMER) have small N and lower per-class recall — expected and explicitly reported.
- **Task 2 (Sentiment from structure):** Structural features (emoji, length, punctuation, post type, author persona) provide signal above the 33% macro-F1 random baseline. The gap to 1.0 represents what only text semantics can provide. This model is deployable without any Vertex/LLM call.
- **Task 3 (Engagement regression):** Post metadata explains a moderate portion of engagement variance at log scale. Coefficients are the main output — R² should not be over-interpreted at N=1,493.
- **Task 4 (Cross-tabs):** Sentiment distributions differ significantly across personas (chi², Kruskal-Wallis). Content-form preference also varies by persona.

### Sampling & imbalance caveats (§3 of modeling plan)
- **B is not a uniform sample:** Sentiment labels (B) cover 487k of 499k IG comments, prioritising media-bearing posts. B-derived rates are not unbiased population estimates of IG sentiment.
- **Sentiment is 81% positive:** Accuracy is a misleading metric. All task results use macro-F1 / balanced accuracy / per-class recall.
- **Persona classes are severely imbalanced** (THE_TAGGER 11.6k vs THE_HATER 164). `class_weight="balanced"` and stratified splits were used throughout.
- **posts_model has only 1,493 rows.** Regression results are descriptive; R² estimates are noisy.